# Propensity Score Matching (simple version)

**Project: EconoCausal**

**The question:** before the campaign, did the customers who got an email look like the customers who got no email?

**Analogy:** if you give one classroom a learning game, you first check that both classrooms started with similar skills. Here we do the same check for customers.

**What this notebook does**
1. Load the Hillstrom data and keep 70% for training (the other 30% is never touched).
2. Use only information known *before* the email (past spending, region, channel, etc.).
3. Predict each customer's chance of getting the email (the *propensity score*).
4. Match every email customer to the most similar no-email customer.
5. Check whether the groups look more alike (balance check).

**Important:** the data has three groups: **Men's email, Women's email, No email**. We compare each email group against No email separately. The outcomes (visit, conversion, spend) are never used here.

## Step 1: Load the data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

SEED = 42

# Find the project folder (the one that contains data/raw/hillstrom.csv)
folders = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in folders if (p / "data/raw/hillstrom.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open the project folder that contains data/raw/hillstrom.csv")

OUTPUT = ROOT / "results/propensity_matching"   # results are saved here
OUTPUT.mkdir(parents=True, exist_ok=True)

data = pd.read_csv(ROOT / "data/raw/hillstrom.csv")
print("Rows and columns:", data.shape)

## Step 2: Clean and split the data

- Give each group a number: No email = 0, Men's = 1, Women's = 2.
- Fix a spelling mistake in the data (`Surburban` becomes `Suburban`).
- Keep 70% for training and set 30% aside. We use the same split and seed (42) as the preprocessing notebook, so the 30% stays untouched.

In [ ]:
data["treatment"] = data["segment"].map({"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2})
data["zip_code"] = data["zip_code"].replace({"Surburban": "Suburban"})

train, holdout = train_test_split(
    data, test_size=0.30, random_state=SEED, stratify=data["treatment"]
)
train = train.copy()

print(train["segment"].value_counts())

## Step 3: Pick the "before the email" features

We only use things the company already knew before sending the email:

| Feature | Meaning |
|---|---|
| `recency` | months since last purchase |
| `history` | past spending |
| `mens`, `womens` | bought men's / women's items before |
| `newbie` | new customer |
| `zip_code`, `channel` | area type and how they buy |

We also add three simple helper columns: `history_log1p` (spending on a smaller scale so big spenders do not dominate), `prior_interest_count` (how many categories they bought from), and `recency_band` (months grouped into 1-4, 5-8, 9-12).

Text columns are turned into 0/1 columns (this is called dummy encoding).

In [ ]:
train["history_log1p"] = np.log1p(train["history"])
train["prior_interest_count"] = train["mens"] + train["womens"]
train["recency_band"] = pd.cut(train["recency"], bins=[0, 4, 8, 12],
                               labels=["1-4 months", "5-8 months", "9-12 months"])

number_columns = ["recency", "history", "mens", "womens", "newbie",
                  "history_log1p", "prior_interest_count"]
text_columns = ["zip_code", "channel", "recency_band"]

features = pd.get_dummies(train[number_columns + text_columns],
                          columns=text_columns, dtype=float)

print("Features used for matching:")
print(list(features.columns))

## Step 4: The matching function

For one email group versus No email, the function below does four things:

1. **Propensity score:** a logistic regression predicts each customer's chance of getting the email. In a randomized test this should be close to 50%, because it was a coin flip.
2. **Caliper:** a maximum allowed gap between two scores (0.2 x the spread of the scores). If the closest match is further away than this, the customer is left unmatched.
3. **Matching:** each email customer is paired with the nearest no-email customer. A no-email customer can be used more than once (matching *with replacement*).
4. **Balance check (SMD):** compares the average of every feature in the two groups, before and after matching.

> **SMD** (standardized mean difference) = (email average - no-email average) / typical spread.
> **Close to 0 is good. Under 0.1 is the usual rule of thumb.**

In [ ]:
def smd(group_a, group_b, spread):
    """Standardized mean difference for every feature."""
    difference = group_a.mean(axis=0) - group_b.mean(axis=0)
    return np.divide(difference, spread, out=np.zeros_like(difference), where=spread > 0)


def run_matching(arm, label):
    # Keep only this email group and the No email group
    pair = train[train["treatment"].isin([0, arm])]
    x = features.loc[pair.index].to_numpy()
    got_email = (pair["treatment"] == arm).astype(int).to_numpy()   # 1 = email, 0 = no email

    # 1) Propensity score (chance of getting the email)
    scaled = StandardScaler().fit_transform(x)
    model = LogisticRegression(max_iter=2000, random_state=SEED).fit(scaled, got_email)
    scores = model.predict_proba(scaled)[:, 1]
    safe = np.clip(scores, 1e-6, 1 - 1e-6)          # avoids divide-by-zero, removes no rows
    logits = np.log(safe / (1 - safe))              # matching is done on this scale

    # 2) Caliper = maximum allowed gap
    spread = np.sqrt((logits[got_email == 1].var(ddof=1) + logits[got_email == 0].var(ddof=1)) / 2)
    caliper = 0.2 * spread

    # 3) Match each email customer to the closest no-email customer
    email_rows = np.flatnonzero(got_email == 1)
    control_rows = np.flatnonzero(got_email == 0)
    gap, nearest = cKDTree(logits[control_rows, None]).query(logits[email_rows, None])
    keep = gap <= caliper
    matched_email = email_rows[keep]
    matched_control = control_rows[nearest[keep]]

    # 4) Balance before and after matching (same "ruler" for both)
    ruler = np.sqrt((x[got_email == 1].var(axis=0, ddof=1) + x[got_email == 0].var(axis=0, ddof=1)) / 2)
    before = smd(x[got_email == 1], x[got_email == 0], ruler)
    after = smd(x[matched_email], x[matched_control], ruler)

    balance = pd.DataFrame({"feature": features.columns, "smd_before": before, "smd_after": after})

    summary = {
        "group": label,
        "email_customers": int(got_email.sum()),
        "matched": int(len(matched_email)),
        "left_out": int(got_email.sum() - len(matched_email)),
        "unique_controls_used": int(len(np.unique(matched_control))),
        "worst_smd_before": float(np.abs(before).max()),
        "worst_smd_after": float(np.abs(after).max()),
    }
    return summary, balance, scores, got_email

## Step 5: Run it for both email groups

In [ ]:
results = {}
summaries = []

for arm, label in [(1, "Mens E-Mail"), (2, "Womens E-Mail")]:
    summary, balance, scores, got_email = run_matching(arm, label)
    results[label] = {"balance": balance, "scores": scores, "got_email": got_email}
    summaries.append(summary)
    balance.to_csv(OUTPUT / f"{label.split()[0].lower()}_balance.csv", index=False)

summary_table = pd.DataFrame(summaries).set_index("group")
summary_table.to_csv(OUTPUT / "matching_summary.csv")
summary_table.round(4)

**How to read this table**
- `matched` / `left_out`: how many email customers found a match (most should).
- `unique_controls_used`: how many different no-email customers were used. It is lower than `matched` because some were reused.
- `worst_smd_before` / `worst_smd_after`: the largest difference between the groups on any feature. **Under 0.1 means the groups look similar.**

## Step 6: Pictures

- **Left:** the two groups' propensity scores. If the histograms overlap a lot, customers in both groups are comparable.
- **Right:** one dot per feature. Dots near 0 mean the groups are alike. The dashed line is the 0.1 limit.

The groups may already be very similar *before* matching, because this was a randomized experiment. So the "after" mark does not always improve.

In [ ]:
for label, r in results.items():
    fig, (left, right) = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

    # Left: overlap of propensity scores
    scores, got_email = r["scores"], r["got_email"]
    bins = np.linspace(scores.min(), scores.max(), 31)
    left.hist(scores[got_email == 0], bins=bins, alpha=0.5, density=True, label="No email")
    left.hist(scores[got_email == 1], bins=bins, alpha=0.5, density=True, label=label)
    left.axvline(0.5, color="black", linestyle="--", label="Expected (50%)")
    left.set(title="Do the two groups overlap?", xlabel="Chance of getting the email", ylabel="Density")
    left.legend()

    # Right: balance for each feature
    b = r["balance"]
    y = np.arange(len(b))
    right.scatter(b["smd_before"].abs(), y, label="Before matching")
    right.scatter(b["smd_after"].abs(), y, marker="x", label="After matching")
    right.axvline(0.1, color="grey", linestyle="--", label="0.1 limit")
    right.set_yticks(y)
    right.set_yticklabels(b["feature"])
    right.set(title="How different are the groups?", xlabel="Absolute SMD (smaller is better)")
    right.legend()

    fig.suptitle(f"{label} vs No E-Mail")
    fig.savefig(OUTPUT / f"{label.split()[0].lower()}_diagnostics.png", dpi=130)
    plt.show()

## Step 7: Plain-English result

In [ ]:
for row in summaries:
    share = 100 * row["matched"] / row["email_customers"]
    good = row["worst_smd_after"] < 0.1
    print(f"{row['group']}:")
    print(f"  - {row['matched']:,} of {row['email_customers']:,} email customers matched ({share:.2f}%).")
    print(f"  - Biggest difference between groups after matching: {row['worst_smd_after']:.3f}"
          f" (before: {row['worst_smd_before']:.3f}).")
    print(f"  - Every feature under 0.1? {'YES, the groups look similar.' if good else 'NO, check the plot.'}")
print()
print("Saved tables and pictures to:", OUTPUT)

## What to tell your manager

"We checked whether the email and no-email customers had similar past behaviour before the campaign. We matched each email customer to a similar no-email customer and measured the differences before and after. The groups were already very similar, which is what we expect from a randomized test. Matching is an extra safety check here."

## Limits and next step

- Good balance only covers the features we **measured**. It cannot rule out hidden differences.
- This notebook does **not** measure the effect of the email. It only checks comparability.
- **Next step:** train the DML model on the original training data from preprocessing (not the matched rows), and use the untouched 30% for uplift / Qini evaluation.